# Phase 7: Text Reconstruction & Visual Export
### Spatial Document Hierarchy Assembly, Plaintext UTF-8 Export & HUD Overlays
---
### What is Phase 7 and why do we need it?
In Phase 6, our shallow MLP classified every isolated character into a predicted letter or digit with a confidence score. But an OCR system cannot stop at a jumble of disconnected characters!

**Phase 7 turns isolated character predictions into real, readable digital documents:**
1. **Spatial Text Assembly**:
   - We traverse the natural reading hierarchy: **Top-to-Bottom for Lines** and **Left-to-Right for Words and Characters**.
   - We assemble characters into words, insert standard ASCII spaces (`' '`) between words, and insert newlines (`'\n'`) between lines.
2. **Dynamic Paragraph Break Detection**:
   - When the vertical gap between consecutive lines exceeds $1.8 \times \text{median line height}$, the system detects an intentional paragraph break and inserts a double newline (`'\n\n'`).
3. **Lossless UTF-8 Plaintext Export**:
   - We save the digital text into `outputs/text/<name>_recognized.txt` using explicit UTF-8 encoding.
4. **High-Resolution Diagnostic Visual HUD Overlays**:
   - We draw **Green bounding boxes** around words and **Cyan bounding boxes** around characters.
   - We render **Solid Background Label Pills** showing the predicted character and confidence score (e.g., `A 98%`) so text remains 100% readable over any desk or paper texture.
   - We render a **Top Telemetry Banner** displaying line count, word count, character count, and overall confidence.

In [ ]:
# Step 1: Import required libraries and Phase 7 modules
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add Model directory to Python path
module_path = Path("../Model").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from ingestion import ingest_image
from preprocessing import preprocess_image
from segmentation import segment_document
from normalization import normalize_all_characters
from features import extract_features_batch
from predict import predict_batch
from reconstruction import (
    reconstruct_text,
    export_text_file,
    reconstruct_and_export,
    ReconstructionEngine,
)
from visualization import (
    render_annotated_document,
    render_side_by_side_comparison,
    VisualizationEngine,
)
from main import run_pipeline
import config

print("All Phase 7 Reconstruction and Visualization modules loaded successfully!")

---
## Step 2: Run End-to-End Recognition on Sample Photograph
Let's execute the complete pipeline on our test photograph `IMG_20261021_212517.jpg` using the master `run_pipeline` engine.

In [ ]:
# Locate sample photograph
sample_file = Path("../data/samples/IMG_20261021_212517.jpg")
if not sample_file.exists():
    sample_file = Path("../data/samples/sample_camera_capture_synth.png")

# Execute full 7-phase pipeline
pipeline_result = run_pipeline(
    input_path=sample_file,
    output_dir="../Model/outputs",
    debug=True,
    quiet=False,
)

print(f"Pipeline execution finished in {pipeline_result.runtime_seconds:.2f} seconds!")

---
## Step 3: Inspect Reconstructed Text & UTF-8 File
Let's inspect the digital text produced by the reconstruction engine and verify that the `.txt` file was created on disk.

In [ ]:
rec = pipeline_result.reconstruction

print("=== Reconstructed Text Metrics ===")
print(f"Total Lines Recognized      : {rec.total_lines}")
print(f"Total Words Formed          : {rec.total_words}")
print(f"Total Characters Recognized : {rec.total_characters}")
print(f"Mean Recognition Confidence : {rec.mean_confidence * 100:.2f}%")
print(f"Exported Text File Path     : {rec.output_file_path}")
print("-" * 50)
print("Reconstructed Document Text Content:")
print("-" * 50)

# Read back the file directly from disk to verify UTF-8 integrity
with open(rec.output_file_path, "r", encoding="utf-8") as f:
    saved_text = f.read()

print(saved_text)
print("-" * 50)

---
## Step 4: Display Annotated Document Visual Canvas
Let's visualize the final output image with color-coded bounding boxes and readable character tags.

In [ ]:
# Load generated annotated image
annotated_bgr = cv2.imread(str(pipeline_result.output_annotated_path))
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 8))
plt.imshow(annotated_rgb)
plt.title("Phase 7: Annotated Document Visual HUD Overlay", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

---
## Step 5: Side-by-Side Comparison (Original vs Recognized)
Here we compare the untouched camera photograph side-by-side with the annotated computer vision result.

In [ ]:
# Generate side-by-side comparison
comparison_bgr = render_side_by_side_comparison(
    pipeline_result.ingestion.untouched_image,
    pipeline_result.annotated_image,
)
comparison_rgb = cv2.cvtColor(comparison_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16, 8))
plt.imshow(comparison_rgb)
plt.title("Left: Original Input Photo | Right: OCR Recognized & Annotated", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

---
## Step 6: Command Line Interface (CLI) Usage
Our entire system can also be executed directly from the terminal:

```powershell
# Standard execution on any photograph
python main.py --input data/samples/IMG_20261021_212517.jpg

# Force perspective deskewing and save intermediate debug masks
python main.py --input data/samples/IMG_20261021_212517.jpg --warp --debug

# Run automated unit tests across all 7 phases
pytest Model/tests/ -v
```

---
## Quick Professor & Viva Cheat Sheet (Phase 7)

> **Q1: How does the system determine word spacing and line breaks?**  
> *Answer:* We traverse the spatial hierarchy constructed in Phase 3. Words belonging to the same text line are sorted left-to-right by horizontal coordinate and joined with a single ASCII space (`' '`). Consecutive lines ordered top-to-bottom are separated by newline characters (`'\n'`).
>
> **Q2: How are paragraph breaks detected?**  
> *Answer:* We calculate the vertical distance (gap) between consecutive text lines. If the gap exceeds $1.8 \times \text{median line height}$, the system detects an intentional paragraph break and inserts a double newline (`'\n\n'`).
>
> **Q3: Why is UTF-8 encoding specifically required for text export?**  
> *Answer:* UTF-8 is the universal standard character encoding for digital text. It guarantees lossless text preservation across Windows, Linux, and macOS without encoding artifacts or symbol corruption.
>
> **Q4: Why are solid background pills rendered behind character HUD labels?**  
> *Answer:* Real camera photos have varied paper tones, shadows, wood desks, or creases. Rendering a high-contrast dark solid rectangle behind character text ensures labels remain 100% legible under any visual conditions.
>
> **Q5: What is the total processing latency of the entire pipeline?**  
> *Answer:* Because our entire architecture is strictly classical computer vision (OpenCV) and a lightweight shallow MLP (39,067 parameters), the end-to-end runtime is under 2 seconds on standard CPU hardware, with zero GPU requirement.